# Bicycle Accident Analysis — Paderborn 2024

This notebook analyses bicycle accident data from the **Destatis Unfallatlas** and enriches it with OpenStreetMap (OSM) data (schools and streetlights) to explore spatial patterns of accidents in Paderborn.

In [44]:
!pip install folium requests pandas -q

import requests
import pandas as pd
import folium


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\MoZa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Load accident data

Read the CSV from Destatis. The file uses **semicolons** as separators (German format), so we pass `sep=';'`.

In [45]:
path = r"C:\Users\MoZa\OneDrive - Universität Paderborn\0_UPB\6\Data Viz\csv\Unfallorte2024_LinRef.csv"

bike_acci = pd.read_csv(path, sep=';')


C:\Users\MoZa\AppData\Local\Temp\ipykernel_30556\3345210901.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  bike_acci = pd.read_csv(path, sep=';')


## 2. Explore the raw data

Check column names, data types, and a preview to understand the structure before any cleaning.

In [59]:
# Always do this before cleaning — understand what you have
print(bike_acci.columns)

print(bike_acci.dtypes)

Index(['OID_', 'UIDENTSTLAE', 'ULAND', 'UREGBEZ', 'UKREIS', 'UGEMEINDE',
       'UJAHR', 'UMONAT', 'USTUNDE', 'UWOCHENTAG', 'UKATEGORIE', 'UART',
       'UTYP1', 'ULICHTVERH', 'IstStrassenzustand', 'IstRad', 'IstPKW',
       'IstFuss', 'IstKrad', 'IstGkfz', 'IstSonstige', 'LINREFX', 'LINREFY',
       'lon', 'lat', 'PLST'],
      dtype='object')
OID_                    int64
UIDENTSTLAE            object
ULAND                   int64
UREGBEZ                 int64
UKREIS                  int64
UGEMEINDE               int64
UJAHR                   int64
UMONAT                  int64
USTUNDE                 int64
UWOCHENTAG              int64
UKATEGORIE              int64
UART                    int64
UTYP1                   int64
ULICHTVERH              int64
IstStrassenzustand      int64
IstRad                  int64
IstPKW                  int64
IstFuss                 int64
IstKrad                 int64
IstGkfz                 int64
IstSonstige             int64
LINREFX                

In [ ]:
pd.set_option('display.max_columns', None)
bike_acci.head()


,OID_,UIDENTSTLAE,ULAND,UREGBEZ,UKREIS,UGEMEINDE,UJAHR,UMONAT,USTUNDE,UWOCHENTAG,UKATEGORIE,UART,UTYP1,ULICHTVERH,IstStrassenzustand,IstRad,IstPKW,IstFuss,IstKrad,IstGkfz,IstSonstige,LINREFX,LINREFY,lon,lat,PLST
25844,25845,5240112411132780673,5,7,74,32,2024,1,14,6,3,0,3,0,1,1,0,0,1,0,0,"480973,502999999560416","5731161,901000000536442",8.724498,51.731054,1
26916,26917,5240203411232530955,5,7,74,32,2024,2,8,7,3,7,1,0,1,1,0,0,0,0,0,"480331,287800000049174","5735749,133099999278784",8.714939,51.772277,1
27912,27913,5240123411232117788,5,7,74,32,2024,1,7,3,2,5,3,1,0,1,1,0,0,0,0,"480184,146900000050664","5736187,297299999743700",8.712782,51.776211,1
30309,30310,5240212411232825939,5,7,74,32,2024,2,15,2,3,0,1,0,1,1,0,0,0,0,0,"476666,576999999582767","5734099,529999999329448",8.661938,51.757304,1
30722,30723,5240318411132052357,5,7,74,32,2024,3,8,2,3,3,6,0,0,1,0,0,0,0,1,"484280,855000000447035","5733726,371999999508262",8.772272,51.754215,1


## 3. Clean and standardise coordinates

The coordinate columns (`XGCSWGS84`, `YGCSWGS84`) use German decimal commas. We rename them to `lat`/`lon` and convert to proper floats.

In [ ]:
# Rename coordinate columns to standard names
bike_acci = bike_acci.rename(columns={
    "XGCSWGS84": "lon",
    "YGCSWGS84": "lat"
})

# Fix decimal separator (German CSVs use comma instead of dot)
bike_acci["lat"] = bike_acci["lat"].astype(str).str.replace(",", ".").astype(float)
bike_acci["lon"] = bike_acci["lon"].astype(str).str.replace(",", ".").astype(float)

# Filter: only bicycle accidents
bike_acci = bike_acci[bike_acci["IstRad"] == 1].copy()
print(f"Total accidents: {len(bike_acci)}")
print(f"Bicycle accidents: {len(bike_acci)}")

# Filter: only Paderborn bounding box (rough coordinates)
# Paderborn is roughly between these coordinates
lat_min, lat_max = 51.65, 51.78
lon_min, lon_max = 8.65, 8.85

bike_acci = bike_acci[
    (bike_acci["lat"] >= lat_min) & (bike_acci["lat"] <= lat_max) &
    (bike_acci["lon"] >= lon_min) & (bike_acci["lon"] <= lon_max)
].copy()

print(f"Bicycle accidents in Paderborn: {len(bike_acci)}")
bike_acci.head()


Total accidents: 86543
Bicycle accidents: 86543
Bicycle accidents in Paderborn: 246


,OID_,UIDENTSTLAE,ULAND,UREGBEZ,UKREIS,UGEMEINDE,UJAHR,UMONAT,USTUNDE,UWOCHENTAG,...,IstPKW,IstFuss,IstKrad,IstGkfz,IstSonstige,LINREFX,LINREFY,lon,lat,PLST
25844,25845,5240112411132780673,5,7,74,32,2024,1,14,6,...,0,0,1,0,0,"480973,502999999560416","5731161,901000000536442",8.724498,51.731054,1
26916,26917,5240203411232530955,5,7,74,32,2024,2,8,7,...,0,0,0,0,0,"480331,287800000049174","5735749,133099999278784",8.714939,51.772277,1
27912,27913,5240123411232117788,5,7,74,32,2024,1,7,3,...,1,0,0,0,0,"480184,146900000050664","5736187,297299999743700",8.712782,51.776211,1
30309,30310,5240212411232825939,5,7,74,32,2024,2,15,2,...,0,0,0,0,0,"476666,576999999582767","5734099,529999999329448",8.661938,51.757304,1
30722,30723,5240318411132052357,5,7,74,32,2024,3,8,2,...,0,0,0,0,1,"484280,855000000447035","5733726,371999999508262",8.772272,51.754215,1


## 4. Add readable labels

Map numeric codes (severity, light conditions, weekday) to human-readable strings so charts and popups are self-explanatory.

In [60]:
bike_acci["severity"] = bike_acci["UKATEGORIE"].map({
    1: "Fatal", 2: "Serious injury", 3: "Light injury"
})
bike_acci["light"] = bike_acci["ULICHTVERH"].map({
    0: "Daylight", 1: "Dusk/dawn", 2: "Dark"
})
bike_acci["weekday"] = bike_acci["UWOCHENTAG"].map({
    1: "Sunday", 2: "Monday", 3: "Tuesday",
    4: "Wednesday", 5: "Thursday", 6: "Friday", 7: "Saturday"
})

## 5. Flag accidents near schools

Mark each accident as near a school if it falls within ~200 m (≈ 0.002°) of any school location. This helper function does a simple bounding-box check.

In [62]:
# Flag accidents within ~200m using a coordinate box
OFFSET = 0.002  # roughly 200 meters in degrees

def is_near_school_simple(acc_lat, acc_lon, schools_df):
    return any(
        (abs(acc_lat - s["lat"]) < OFFSET) and 
        (abs(acc_lon - s["lon"]) < OFFSET)
        for _, s in schools_df.iterrows()
    )

bike_acci["near_school"] = bike_acci.apply(
    lambda row: is_near_school_simple(row["lat"], row["lon"], df_schools),
    axis=1
)

## 6. Fetch OpenStreetMap data

Query the **Overpass API** to get locations of schools and streetlights within Paderborn's administrative boundary. Results are returned as a clean DataFrame with `lat`, `lon`, and `name`.

In [28]:
def fetch_osm(query):
    """Send an Overpass QL query and return a clean DataFrame."""
    headers = {"User-Agent": "DataVis-Assignment/1.0 (student project)"}
    for url in [
        "https://overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
    ]:
        try:
            response = requests.post(url, data={"data": query}, headers=headers, timeout=60)
            response.raise_for_status()
            break
        except Exception as e:
            last_error = e
    else:
        raise last_error

    elements = response.json()["elements"]
    rows = []
    for el in elements:
        lat = el.get("lat") or el.get("center", {}).get("lat")
        lon = el.get("lon") or el.get("center", {}).get("lon")
        if lat and lon:
            row = {"lat": lat, "lon": lon}
            row.update(el.get("tags", {}))
            rows.append(row)

    return pd.DataFrame(rows)

### 6a. Fetch schools in Paderborn

In [29]:
school_query = """
[out:json][timeout:60];
area["name"="Paderborn"]["boundary"="administrative"]->.city;
(
  node["amenity"="school"](area.city);
  way["amenity"="school"](area.city);
);
out center;
"""

df_schools = fetch_osm(school_query)
df_schools["type"] = "school"

# Keep only useful columns
df_schools = df_schools[["lat", "lon", "name", "type"]].fillna("Unknown")
print(f"Found {len(df_schools)} schools")
df_schools.head()

Found 77 schools


,lat,lon,name,type
0,51.717192,8.731331,Riemeke Grundschule,school
1,51.745447,8.712306,Realschule Schloß Neuhaus,school
2,51.717463,8.731585,Lutherschule West,school
3,51.720142,8.740261,"Gregor-Mendel-Berufskolleg, Berufsfeld Agrarwi...",school
4,51.729641,8.776737,Mettenmeier Geschäftsbereich Bildung,school


### 6b. Fetch streetlights

In [30]:
light_query = """
[out:json][timeout:60];
area["name"="Paderborn"]["boundary"="administrative"]->.city;
(
  node["highway"="street_lamp"](area.city);
);
out center;
"""

df_lights = fetch_osm(light_query)
df_lights["type"] = "streetlight"
df_lights["name"] = "Street lamp"

df_lights = df_lights[["lat", "lon", "name", "type"]].fillna("Unknown")
print(f"Found {len(df_lights)} streetlights")
df_lights.head()

Found 2258 streetlights


,lat,lon,name,type
0,51.710010,8.767271,Street lamp,streetlight
1,51.710247,8.767560,Street lamp,streetlight
2,51.708053,8.771479,Street lamp,streetlight
3,51.707682,8.771767,Street lamp,streetlight
4,51.707661,8.771783,Street lamp,streetlight


### 6c. Combine schools and streetlights

Merge both OSM datasets into one DataFrame for easy filtering and plotting.

In [35]:
df_osm = pd.concat([df_schools, df_lights], ignore_index=True)
print(df_osm["type"].value_counts())
df_osm.tail(10)

type
streetlight    2258
school           77
Name: count, dtype: int64


,lat,lon,name,type
2325,51.714666,8.752595,Street lamp,streetlight
2326,51.714580,8.748041,Street lamp,streetlight
2327,51.717046,8.746662,Street lamp,streetlight
2328,51.706203,8.723716,Street lamp,streetlight
2329,51.706324,8.723855,Street lamp,streetlight
2330,51.693990,8.681384,Street lamp,streetlight
2331,51.693758,8.681875,Street lamp,streetlight
2332,51.693808,8.681566,Street lamp,streetlight
2333,51.694133,8.681610,Street lamp,streetlight
2334,51.714001,8.741228,Street lamp,streetlight


## 7. Interactive map

Plot schools (blue markers) and streetlights (orange circles) on an interactive Folium map centred on Paderborn.

In [38]:
import folium
from IPython.display import display

m = folium.Map(location=[51.718, 8.757], zoom_start=13)

for _, row in df_schools.iterrows():
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=row["name"],
        tooltip=row["name"],
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(m)

for _, row in df_lights.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color="orange",
        fill=True,
        fill_opacity=0.6,
        tooltip="Street lamp"
    ).add_to(m)

display(m)   # shows inline in VS Code notebook